In [1]:
import requests
import pandas as pd
from mainnet_launch.constants import AUTO_ETH
from web3 import Web3


def paginate_query(api_url: str, query: str, variables: dict, data_col: str) -> list[dict]:
    """
    Helper to page through a GraphQL connection using `first`/`skip`.

    :param api_url: GraphQL endpoint
    :param query: The GraphQL query string, expecting $first and $skip variables
    :param variables: Base variables (e.g. {"autoEthAddress": "..."}).
                      first/skip will be merged in each loop.
    :param batch_size: Number of items to fetch per request.
    :return: List of result dicts
    """

    all_records = []
    skip = 0

    while True:
        vars_with_pagination = {**variables, "first": 500, "skip": skip}
        resp = requests.post(api_url, json={"query": query, "variables": vars_with_pagination})
        resp.raise_for_status()
        batch = resp.json()["data"][data_col]

        if not batch:
            break

        all_records.extend(batch)
        skip += 500

    return all_records


def _get_subgraph_api(chain):
    if chain == "eth":
        api_url = "https://subgraph.satsuma-prod.com/108d48ba91e3/tokemak/v2-gen3-eth-mainnet/api"
    elif chain == "base":
        api_url = "https://subgraph.satsuma-prod.com/108d48ba91e3/tokemak/v2-gen3-base-mainnet/api"
    else:
        raise ValueError("bad chain", chain)

    return api_url


def fetch_autopool_rebalance_events_from_subgraph(autopool_eth_addr: str, chain: str) -> list[dict]:
    """
    Fetches all AutopoolRebalances entries for the given autopool.
    """
    subgraph_url = _get_subgraph_api(chain)

    query = """
    query($autoEthAddress: String!, $first: Int!, $skip: Int!) {
      autopoolRebalances(
        first: $first,
        skip: $skip,
        orderBy: id,
        orderDirection: desc,
        where: { autopool: $autoEthAddress }
      ) {
        transactionHash
        timestamp
        blockNumber

        tokenInAmount
        tokenOut {
        id
        decimals
        
        }        
        destinationInAddress

        tokenIn {
        id
        decimals
        
        }   

        tokenOutAmount
        destinationOutAddress
        
      }
    }
    """

    # Adjust `first` as needed; paginate_query will loop over skip increments of `first`
    df = pd.DataFrame.from_records(
        paginate_query(
            subgraph_url,
            query,
            variables={"autoEthAddress": autopool_eth_addr.lower(), "first": 1000, "skip": 0},
            data_col="autopoolRebalances",
        )
    )

    df["tokenInAddress"] = df["tokenIn"].apply(lambda x: x["id"])
    df["tokenOutAddress"] = df["tokenOut"].apply(lambda x: x["id"])

    df["tokenOutAmount"] = df.apply(
        lambda row: int(row["tokenOutAmount"]) / (10 ** int(row["tokenOut"]["decimals"])), axis=1
    )
    df["tokenInAmount"] = df.apply(
        lambda row: int(row["tokenInAmount"]) / (10 ** int(row["tokenIn"]["decimals"])), axis=1
    )

    df["blockNumber"] = df["blockNumber"].astype(int)

    df["destinationInAddress"] = df["destinationInAddress"].apply(lambda x: Web3.toChecksumAddress(x))
    df["tokenInAddress"] = df["tokenInAddress"].apply(lambda x: Web3.toChecksumAddress(x))

    df["destinationOutAddress"] = df["destinationOutAddress"].apply(lambda x: Web3.toChecksumAddress(x))
    df["tokenOutAddress"] = df["tokenOutAddress"].apply(lambda x: Web3.toChecksumAddress(x))

    return df


rebalance_event_df = fetch_autopool_rebalance_events_from_subgraph("0xa7569A44f348d3D70d8ad5889e50F78E33d80D35", "eth")
rebalance_event_df

,transactionHash,timestamp,blockNumber,tokenInAmount,tokenOut,destinationInAddress,tokenIn,tokenOutAmount,destinationOutAddress,tokenInAddress,tokenOutAddress
0,0xff8915d0bd9054e7678c4366ae8fd5a0a3cce80943be...,1744988327,22296632,148881.150087,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0xbB2d2dd491204a86ec10a1a6972F940B34fE060e,{'id': '0x4f493b7de8aac7d55f71853688b1f7c8f024...,150000.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x4f493B7dE8aAC7d55F71853688b1F7C8F0243C85,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48
1,0xff3d22a3564c8121f28d7d47feffb8251785ad2c48e4...,1744997471,22297389,148885.474474,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0xbB2d2dd491204a86ec10a1a6972F940B34fE060e,{'id': '0x4f493b7de8aac7d55f71853688b1f7c8f024...,150000.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x4f493B7dE8aAC7d55F71853688b1F7C8F0243C85,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48
2,0xfecf659a8d1c06c45010afc47fcefc6405d906a85e84...,1746235043,22399927,320651.575857,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0xC5c95fCad37E466E25e6ecA1977bbF75C0E1004a,{'id': '0xbeef01735c132ada46aa9aa4c54623caa92a...,347222.682150,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0xBEEF01735c132Ada46AA9aA4c54623cAA92A64CB,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48
3,0xfca3e4ad4e6296b2444ed28a8012cda8694ef5d23ce9...,1746116255,22390138,148075.994800,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0x7876F91BB22148345b3De16af9448081E9853830,{'id': '0x9fb7b4477576fe5b32be4c1843afb1e55f25...,170000.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x9Fb7b4477576Fe5B32be4C1843aFB1e55F251B33,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48
4,0xfc37985d3c7182369be30f9dbe69409a8b05d008bde2...,1745111003,22306807,437134.294459,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0xE4545f9dBC30Ccb6Cda6930DDFd69f3D419FcB61,{'id': '0x5c20b550819128074fd538edf79791733cce...,500000.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x5C20B550819128074FD538Edf79791733ccEdd18,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48
...,...,...,...,...,...,...,...,...,...,...,...
264,0x02a5528289efe28718bed54e9c5b0772f16eba69d609...,1744695875,22272381,76347.504659,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0x7583b1589aDD33320366A48A92794D77763FAE9e,{'id': '0x390f3595bca2df7d23783dfd126427cceb99...,77679.210200,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x390f3595bCa2Df7d23783dFd126427CCeb997BF4,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48
265,0x02065d1b16c7c693afe6358f38647b921a095d6876ec...,1745911055,22373171,26611.724622,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0x7583b1589aDD33320366A48A92794D77763FAE9e,{'id': '0x390f3595bca2df7d23783dfd126427cceb99...,27068.157900,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x390f3595bCa2Df7d23783dFd126427CCeb997BF4,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48
266,0x012e26a2633c10388ffc3ee1a2a95c9569b54ff4f918...,1744556327,22260800,140053.102440,{'id': '0x4f493b7de8aac7d55f71853688b1f7c8f024...,0xC099899d0278CE83976218Cbe58D01dD382dcA32,{'id': '0x57064f49ad7123c92560882a45518374ad98...,149036.479162,0xbB2d2dd491204a86ec10a1a6972F940B34fE060e,0x57064F49Ad7123C92560882a45518374ad982e85,0x4f493B7dE8aAC7d55F71853688b1F7C8F0243C85
267,0x00ff23899cd9bb0075ebe5ac23a3faa815088ddd32ab...,1744906595,22289850,148973.430940,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0xbB2d2dd491204a86ec10a1a6972F940B34fE060e,{'id': '0x4f493b7de8aac7d55f71853688b1f7c8f024...,150000.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x4f493B7dE8aAC7d55F71853688b1F7C8F0243C85,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48


In [ ]:
from mainnet_launch.database.schema.full import Blocks, DestinationTokenValues, DestinationStates, Destinations
from mainnet_launch.database.schema.postgres_operations import (
    merge_tables_as_df,
    TableSelector,
)
from mainnet_launch.database.schema.ensure_tables_are_current.update_destinations_states_table import (
    _add_new_destination_states_to_db,
)

_add_new_destination_states_to_db(rebalance_event_df["blockNumber"], AUTO_ETH.chain)

destination_states_df = merge_tables_as_df(
    [
        TableSelector(DestinationStates, [DestinationStates.block, DestinationStates.lp_token_spot_price]),
        TableSelector(
            Destinations,
            [Destinations.underlying, Destinations.underlying_symbol],
            join_on=(DestinationStates.chain_id == DestinationStates.chain_id)
            & (Destinations.destination_vault_address == DestinationStates.destination_vault_address),
        ),
    ]
)
destination_states_df

2025-05-05 13:25:09,445 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-05-05 13:25:09,446 INFO sqlalchemy.engine.Engine 
            SELECT *
            FROM destination_states
            WHERE destination_states.chain_id = 1
        
2025-05-05 13:25:09,447 INFO sqlalchemy.engine.Engine [cached since 513.5s ago] {}
2025-05-05 13:25:10,661 INFO sqlalchemy.engine.Engine COMMIT
2025-05-05 13:25:10,782 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-05-05 13:25:10,783 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s, %(param_2)s, %(param_3)s, %(param_4)s, %(param_5)s]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class.oid) AND pg_catalog.pg_namespace.nspname != %(nspname_1)s
2025-05-05 13:25:10,783 INFO sqlalchemy.engine.Engine [ca

,block,lp_token_spot_price,underlying,underlying_symbol
0,20759464,1.036516,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,B-rETH-STABLE
1,20766617,1.036121,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,B-rETH-STABLE
2,20773761,1.036456,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,B-rETH-STABLE
3,20780916,1.036477,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,B-rETH-STABLE
4,20788084,1.036502,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,B-rETH-STABLE
...,...,...,...,...
15195,22272381,1.000000,0x35911af1B570E26f668905595dEd133D01CD3E5a,dineroETH
15196,22373171,1.000000,0x35911af1B570E26f668905595dEd133D01CD3E5a,dineroETH
15197,22260800,1.000000,0x35911af1B570E26f668905595dEd133D01CD3E5a,dineroETH
15198,22289850,1.000000,0x35911af1B570E26f668905595dEd133D01CD3E5a,dineroETH


In [38]:
destination_states_df["underlying_symbol"].value_counts()

underlying_symbol
weeth-ng                733
weETH/ezETH/rswETH      733
rsETH / WETH            733
weETH/rETH              733
pxethweth               733
pxsteth                 733
ethx-f                  733
osETH-rETH              733
pxETH/wETH              733
ETHx/wstETH             733
ezETH-WETH-BPT          733
autoETH                 501
balETH                  501
autoLRT                 501
dineroETH               501
OETHCRV-f               501
rsETH / ETHx            464
B-rETH-STABLE           464
osETH/wETH-BPT          464
wETHrETH                464
frxeth-ng-f             464
ECLP-wstETH-cbETH       464
ECLP-wstETH-wETH        464
wstETH-WETH-BPT         464
rETH-WETH-BPT           184
ECLP-cbETH-wstETH       184
weETH/wETH              184
vAMM-weETH.base/WETH    184
baseETH                 184
Name: count, dtype: int64

In [34]:
rebalance_event_df["tokenInAddress"].value_counts()

tokenInAddress
0x4f493B7dE8aAC7d55F71853688b1F7C8F0243C85    60
0x635EF0056A597D13863B73825CcA297236578595    37
0x390f3595bCa2Df7d23783dFd126427CCeb997BF4    30
0x5C20B550819128074FD538Edf79791733ccEdd18    28
0x9Fb7b4477576Fe5B32be4C1843aFB1e55F251B33    23
0xfD1627E3f3469C8392C8c3A261D8F0677586e5e1    23
0x4DEcE678ceceb27446b35C672dC7d61F30bAD69E    19
0x5B03CcCAb7BA3010fA5CAd23746cbf0794938e96    18
0x57064F49Ad7123C92560882a45518374ad982e85    14
0x81A2612F6dEA269a6Dd1F6DeAb45C5424EE2c4b7     7
0xdd0f28e19C1780eb6396170735D45153D261490d     4
0x8eB67A509616cd6A7c1B3c8C21D48FF57df3d458     2
0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48     2
0xBEEF01735c132Ada46AA9aA4c54623cAA92A64CB     1
0xb819feeF8F0fcDC268AfE14162983A69f6BF179E     1
Name: count, dtype: int64

In [36]:
destination_states_df[destination_states_df["underlying"] == "0x635EF0056A597D13863B73825CcA297236578595"]

,block,lp_token_spot_price,underlying,underlying_symbol


In [ ]:
df1 = pd.merge(
    rebalance_event_df,
    destination_states_df,
    left_on=["blockNumber", "tokenInAddress"],
    right_on=["block", "underlying"],
    how="left",
)
df1.isna().sum()
# df1 = pd.merge(
#     rebalance_event_df,
#     destination_states_df,
#     left_on=["blockNumber", "destinationInAddress", "tokenInAddress"],
#     right_on=["block", "destination_vault_address", "underlying"],
#     how="left",
# )

transactionHash            0
timestamp                  0
blockNumber                0
tokenInAmount              0
tokenOut                   0
destinationInAddress       0
tokenIn                    0
tokenOutAmount             0
destinationOutAddress      0
tokenInAddress             0
tokenOutAddress            0
block                    269
lp_token_spot_price      269
underlying               269
underlying_symbol        269
dtype: int64

In [6]:
destination_states_df

,destination_vault_address,block,chain_id,incentive_apr,fee_apr,base_apr,points_apr,fee_plus_base_apr,total_apr_in,total_apr_out,...,exchange_name,block_deployed,name,symbol,pool_type,pool,underlying,underlying_symbol,underlying_name,denominated_in
0,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20759464,1,0.041235,0.003997,0.005817,0.0,None,0.046281,0.046281,...,balancer,20756405,Tokemak-Wrapped Ether-Balancer rETH Stable Pool,toke-WETH-B-rETH-STABLE,balMetaStable,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,B-rETH-STABLE,Balancer rETH Stable Pool,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2
1,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20766617,1,0.040938,0.003618,0.005802,0.0,None,0.045987,0.045987,...,balancer,20756405,Tokemak-Wrapped Ether-Balancer rETH Stable Pool,toke-WETH-B-rETH-STABLE,balMetaStable,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,B-rETH-STABLE,Balancer rETH Stable Pool,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2
2,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20773761,1,0.040507,0.003295,0.005848,0.0,None,0.045061,0.045061,...,balancer,20756405,Tokemak-Wrapped Ether-Balancer rETH Stable Pool,toke-WETH-B-rETH-STABLE,balMetaStable,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,B-rETH-STABLE,Balancer rETH Stable Pool,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2
3,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20780916,1,0.040196,0.003022,0.005817,0.0,None,0.044440,0.044440,...,balancer,20756405,Tokemak-Wrapped Ether-Balancer rETH Stable Pool,toke-WETH-B-rETH-STABLE,balMetaStable,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,B-rETH-STABLE,Balancer rETH Stable Pool,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2
4,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20788084,1,0.040448,0.002794,0.005885,0.0,None,0.044524,0.044524,...,balancer,20756405,Tokemak-Wrapped Ether-Balancer rETH Stable Pool,toke-WETH-B-rETH-STABLE,balMetaStable,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,B-rETH-STABLE,Balancer rETH Stable Pool,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15195,0x35911af1B570E26f668905595dEd133D01CD3E5a,22272381,1,0.000000,0.000000,0.000000,0.0,None,0.000000,0.000000,...,tokemak,21718586,dineroETH,dineroETH,idle,0x35911af1B570E26f668905595dEd133D01CD3E5a,0x35911af1B570E26f668905595dEd133D01CD3E5a,dineroETH,dineroETH,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2
15196,0x35911af1B570E26f668905595dEd133D01CD3E5a,22373171,1,0.000000,0.000000,0.000000,0.0,None,0.000000,0.000000,...,tokemak,21718586,dineroETH,dineroETH,idle,0x35911af1B570E26f668905595dEd133D01CD3E5a,0x35911af1B570E26f668905595dEd133D01CD3E5a,dineroETH,dineroETH,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2
15197,0x35911af1B570E26f668905595dEd133D01CD3E5a,22260800,1,0.000000,0.000000,0.000000,0.0,None,0.000000,0.000000,...,tokemak,21718586,dineroETH,dineroETH,idle,0x35911af1B570E26f668905595dEd133D01CD3E5a,0x35911af1B570E26f668905595dEd133D01CD3E5a,dineroETH,dineroETH,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2
15198,0x35911af1B570E26f668905595dEd133D01CD3E5a,22289850,1,0.000000,0.000000,0.000000,0.0,None,0.000000,0.000000,...,tokemak,21718586,dineroETH,dineroETH,idle,0x35911af1B570E26f668905595dEd133D01CD3E5a,0x35911af1B570E26f668905595dEd133D01CD3E5a,dineroETH,dineroETH,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2


In [3]:
rebalance_event_df

,transactionHash,timestamp,blockNumber,tokenInAmount,tokenOut,destinationInAddress,tokenIn,tokenOutAmount,destinationOutAddress,tokenInAddress,tokenOutAddress
0,0xff8915d0bd9054e7678c4366ae8fd5a0a3cce80943be...,1744988327,22296632,148881.150087,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0xbB2d2dd491204a86ec10a1a6972F940B34fE060e,{'id': '0x4f493b7de8aac7d55f71853688b1f7c8f024...,150000.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x4f493B7dE8aAC7d55F71853688b1F7C8F0243C85,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48
1,0xff3d22a3564c8121f28d7d47feffb8251785ad2c48e4...,1744997471,22297389,148885.474474,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0xbB2d2dd491204a86ec10a1a6972F940B34fE060e,{'id': '0x4f493b7de8aac7d55f71853688b1f7c8f024...,150000.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x4f493B7dE8aAC7d55F71853688b1F7C8F0243C85,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48
2,0xfecf659a8d1c06c45010afc47fcefc6405d906a85e84...,1746235043,22399927,320651.575857,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0xC5c95fCad37E466E25e6ecA1977bbF75C0E1004a,{'id': '0xbeef01735c132ada46aa9aa4c54623caa92a...,347222.682150,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0xBEEF01735c132Ada46AA9aA4c54623cAA92A64CB,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48
3,0xfca3e4ad4e6296b2444ed28a8012cda8694ef5d23ce9...,1746116255,22390138,148075.994800,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0x7876F91BB22148345b3De16af9448081E9853830,{'id': '0x9fb7b4477576fe5b32be4c1843afb1e55f25...,170000.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x9Fb7b4477576Fe5B32be4C1843aFB1e55F251B33,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48
4,0xfc37985d3c7182369be30f9dbe69409a8b05d008bde2...,1745111003,22306807,437134.294459,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0xE4545f9dBC30Ccb6Cda6930DDFd69f3D419FcB61,{'id': '0x5c20b550819128074fd538edf79791733cce...,500000.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x5C20B550819128074FD538Edf79791733ccEdd18,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48
...,...,...,...,...,...,...,...,...,...,...,...
264,0x02a5528289efe28718bed54e9c5b0772f16eba69d609...,1744695875,22272381,76347.504659,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0x7583b1589aDD33320366A48A92794D77763FAE9e,{'id': '0x390f3595bca2df7d23783dfd126427cceb99...,77679.210200,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x390f3595bCa2Df7d23783dFd126427CCeb997BF4,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48
265,0x02065d1b16c7c693afe6358f38647b921a095d6876ec...,1745911055,22373171,26611.724622,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0x7583b1589aDD33320366A48A92794D77763FAE9e,{'id': '0x390f3595bca2df7d23783dfd126427cceb99...,27068.157900,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x390f3595bCa2Df7d23783dFd126427CCeb997BF4,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48
266,0x012e26a2633c10388ffc3ee1a2a95c9569b54ff4f918...,1744556327,22260800,140053.102440,{'id': '0x4f493b7de8aac7d55f71853688b1f7c8f024...,0xC099899d0278CE83976218Cbe58D01dD382dcA32,{'id': '0x57064f49ad7123c92560882a45518374ad98...,149036.479162,0xbB2d2dd491204a86ec10a1a6972F940B34fE060e,0x57064F49Ad7123C92560882a45518374ad982e85,0x4f493B7dE8aAC7d55F71853688b1F7C8F0243C85
267,0x00ff23899cd9bb0075ebe5ac23a3faa815088ddd32ab...,1744906595,22289850,148973.430940,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0xbB2d2dd491204a86ec10a1a6972F940B34fE060e,{'id': '0x4f493b7de8aac7d55f71853688b1f7c8f024...,150000.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x4f493B7dE8aAC7d55F71853688b1F7C8F0243C85,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48


In [8]:
rebalance_event_df[["blockNumber", "destinationInAddress", "tokenInAddress"]]

,blockNumber,destinationInAddress,tokenInAddress
0,22296632,0xbB2d2dd491204a86ec10a1a6972F940B34fE060e,0x4f493B7dE8aAC7d55F71853688b1F7C8F0243C85
1,22297389,0xbB2d2dd491204a86ec10a1a6972F940B34fE060e,0x4f493B7dE8aAC7d55F71853688b1F7C8F0243C85
2,22399927,0xC5c95fCad37E466E25e6ecA1977bbF75C0E1004a,0xBEEF01735c132Ada46AA9aA4c54623cAA92A64CB
3,22390138,0x7876F91BB22148345b3De16af9448081E9853830,0x9Fb7b4477576Fe5B32be4C1843aFB1e55F251B33
4,22306807,0xE4545f9dBC30Ccb6Cda6930DDFd69f3D419FcB61,0x5C20B550819128074FD538Edf79791733ccEdd18
...,...,...,...
264,22272381,0x7583b1589aDD33320366A48A92794D77763FAE9e,0x390f3595bCa2Df7d23783dFd126427CCeb997BF4
265,22373171,0x7583b1589aDD33320366A48A92794D77763FAE9e,0x390f3595bCa2Df7d23783dFd126427CCeb997BF4
266,22260800,0xC099899d0278CE83976218Cbe58D01dD382dcA32,0x57064F49Ad7123C92560882a45518374ad982e85
267,22289850,0xbB2d2dd491204a86ec10a1a6972F940B34fE060e,0x4f493B7dE8aAC7d55F71853688b1F7C8F0243C85


In [10]:
destination_states_df

,destination_vault_address,block,chain_id,incentive_apr,fee_apr,base_apr,points_apr,fee_plus_base_apr,total_apr_in,total_apr_out,...,exchange_name,block_deployed,name,symbol,pool_type,pool,underlying,underlying_symbol,underlying_name,denominated_in
0,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20759464,1,0.041235,0.003997,0.005817,0.0,None,0.046281,0.046281,...,balancer,20756405,Tokemak-Wrapped Ether-Balancer rETH Stable Pool,toke-WETH-B-rETH-STABLE,balMetaStable,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,B-rETH-STABLE,Balancer rETH Stable Pool,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2
1,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20766617,1,0.040938,0.003618,0.005802,0.0,None,0.045987,0.045987,...,balancer,20756405,Tokemak-Wrapped Ether-Balancer rETH Stable Pool,toke-WETH-B-rETH-STABLE,balMetaStable,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,B-rETH-STABLE,Balancer rETH Stable Pool,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2
2,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20773761,1,0.040507,0.003295,0.005848,0.0,None,0.045061,0.045061,...,balancer,20756405,Tokemak-Wrapped Ether-Balancer rETH Stable Pool,toke-WETH-B-rETH-STABLE,balMetaStable,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,B-rETH-STABLE,Balancer rETH Stable Pool,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2
3,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20780916,1,0.040196,0.003022,0.005817,0.0,None,0.044440,0.044440,...,balancer,20756405,Tokemak-Wrapped Ether-Balancer rETH Stable Pool,toke-WETH-B-rETH-STABLE,balMetaStable,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,B-rETH-STABLE,Balancer rETH Stable Pool,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2
4,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20788084,1,0.040448,0.002794,0.005885,0.0,None,0.044524,0.044524,...,balancer,20756405,Tokemak-Wrapped Ether-Balancer rETH Stable Pool,toke-WETH-B-rETH-STABLE,balMetaStable,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,B-rETH-STABLE,Balancer rETH Stable Pool,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15195,0x35911af1B570E26f668905595dEd133D01CD3E5a,22272381,1,0.000000,0.000000,0.000000,0.0,None,0.000000,0.000000,...,tokemak,21718586,dineroETH,dineroETH,idle,0x35911af1B570E26f668905595dEd133D01CD3E5a,0x35911af1B570E26f668905595dEd133D01CD3E5a,dineroETH,dineroETH,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2
15196,0x35911af1B570E26f668905595dEd133D01CD3E5a,22373171,1,0.000000,0.000000,0.000000,0.0,None,0.000000,0.000000,...,tokemak,21718586,dineroETH,dineroETH,idle,0x35911af1B570E26f668905595dEd133D01CD3E5a,0x35911af1B570E26f668905595dEd133D01CD3E5a,dineroETH,dineroETH,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2
15197,0x35911af1B570E26f668905595dEd133D01CD3E5a,22260800,1,0.000000,0.000000,0.000000,0.0,None,0.000000,0.000000,...,tokemak,21718586,dineroETH,dineroETH,idle,0x35911af1B570E26f668905595dEd133D01CD3E5a,0x35911af1B570E26f668905595dEd133D01CD3E5a,dineroETH,dineroETH,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2
15198,0x35911af1B570E26f668905595dEd133D01CD3E5a,22289850,1,0.000000,0.000000,0.000000,0.0,None,0.000000,0.000000,...,tokemak,21718586,dineroETH,dineroETH,idle,0x35911af1B570E26f668905595dEd133D01CD3E5a,0x35911af1B570E26f668905595dEd133D01CD3E5a,dineroETH,dineroETH,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2


In [11]:
destination_states_df[["block", "destination_vault_address", "underlying"]]

,block,destination_vault_address,destination_vault_address,underlying
0,20759464,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276
1,20766617,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276
2,20773761,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276
3,20780916,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276
4,20788084,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276
...,...,...,...,...
15195,22272381,0x35911af1B570E26f668905595dEd133D01CD3E5a,0x35911af1B570E26f668905595dEd133D01CD3E5a,0x35911af1B570E26f668905595dEd133D01CD3E5a
15196,22373171,0x35911af1B570E26f668905595dEd133D01CD3E5a,0x35911af1B570E26f668905595dEd133D01CD3E5a,0x35911af1B570E26f668905595dEd133D01CD3E5a
15197,22260800,0x35911af1B570E26f668905595dEd133D01CD3E5a,0x35911af1B570E26f668905595dEd133D01CD3E5a,0x35911af1B570E26f668905595dEd133D01CD3E5a
15198,22289850,0x35911af1B570E26f668905595dEd133D01CD3E5a,0x35911af1B570E26f668905595dEd133D01CD3E5a,0x35911af1B570E26f668905595dEd133D01CD3E5a


In [14]:
destination_states_df = destination_states_df.drop_duplicates()

,transactionHash,timestamp,blockNumber,tokenInAmount,tokenOut,destinationInAddress,tokenIn,tokenOutAmount,destinationOutAddress,tokenInAddress,...,fee_plus_base_apr,total_apr_in,total_apr_out,underlying_token_total_supply,safe_total_supply,price_per_share,price_return,lp_token_spot_price,underlying,underlying_symbol
0,0xff8915d0bd9054e7678c4366ae8fd5a0a3cce80943be...,1744988327,22296632,148881.150087,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0xbB2d2dd491204a86ec10a1a6972F940B34fE060e,{'id': '0x4f493b7de8aac7d55f71853688b1f7c8f024...,150000.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x4f493B7dE8aAC7d55F71853688b1F7C8F0243C85,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0xff3d22a3564c8121f28d7d47feffb8251785ad2c48e4...,1744997471,22297389,148885.474474,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0xbB2d2dd491204a86ec10a1a6972F940B34fE060e,{'id': '0x4f493b7de8aac7d55f71853688b1f7c8f024...,150000.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x4f493B7dE8aAC7d55F71853688b1F7C8F0243C85,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0xfecf659a8d1c06c45010afc47fcefc6405d906a85e84...,1746235043,22399927,320651.575857,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0xC5c95fCad37E466E25e6ecA1977bbF75C0E1004a,{'id': '0xbeef01735c132ada46aa9aa4c54623caa92a...,347222.682150,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0xBEEF01735c132Ada46AA9aA4c54623cAA92A64CB,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0xfca3e4ad4e6296b2444ed28a8012cda8694ef5d23ce9...,1746116255,22390138,148075.994800,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0x7876F91BB22148345b3De16af9448081E9853830,{'id': '0x9fb7b4477576fe5b32be4c1843afb1e55f25...,170000.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x9Fb7b4477576Fe5B32be4C1843aFB1e55F251B33,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0xfc37985d3c7182369be30f9dbe69409a8b05d008bde2...,1745111003,22306807,437134.294459,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0xE4545f9dBC30Ccb6Cda6930DDFd69f3D419FcB61,{'id': '0x5c20b550819128074fd538edf79791733cce...,500000.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x5C20B550819128074FD538Edf79791733ccEdd18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
264,0x02a5528289efe28718bed54e9c5b0772f16eba69d609...,1744695875,22272381,76347.504659,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0x7583b1589aDD33320366A48A92794D77763FAE9e,{'id': '0x390f3595bca2df7d23783dfd126427cceb99...,77679.210200,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x390f3595bCa2Df7d23783dFd126427CCeb997BF4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
265,0x02065d1b16c7c693afe6358f38647b921a095d6876ec...,1745911055,22373171,26611.724622,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0x7583b1589aDD33320366A48A92794D77763FAE9e,{'id': '0x390f3595bca2df7d23783dfd126427cceb99...,27068.157900,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x390f3595bCa2Df7d23783dFd126427CCeb997BF4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
266,0x012e26a2633c10388ffc3ee1a2a95c9569b54ff4f918...,1744556327,22260800,140053.102440,{'id': '0x4f493b7de8aac7d55f71853688b1f7c8f024...,0xC099899d0278CE83976218Cbe58D01dD382dcA32,{'id': '0x57064f49ad7123c92560882a45518374ad98...,149036.479162,0xbB2d2dd491204a86ec10a1a6972F940B34fE060e,0x57064F49Ad7123C92560882a45518374ad982e85,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
267,0x00ff23899cd9bb0075ebe5ac23a3faa815088ddd32ab...,1744906595,22289850,148973.430940,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0xbB2d2dd491204a86ec10a1a6972F940B34fE060e,{'id': '0x4f493b7de8aac7d55f71853688b1f7c8f024...,150000.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x4f493B7dE8aAC7d55F71853688b1F7C8F0243C85,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
rebalance_event_df.head(1).T

,0
transactionHash,0xff8915d0bd9054e7678c4366ae8fd5a0a3cce80943be...
timestamp,1744988327
blockNumber,22296632
tokenInAmount,148881.150087
tokenOut,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...
destinationInAddress,0xbB2d2dd491204a86ec10a1a6972F940B34fE060e
tokenIn,{'id': '0x4f493b7de8aac7d55f71853688b1f7c8f024...
tokenOutAmount,150000.0
destinationOutAddress,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35
tokenInAddress,0x4f493B7dE8aAC7d55F71853688b1F7C8F0243C85


In [ ]:
# 0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48

In [ ]:
token_value_df[
    (token_value_df["block"] == 22296632)
    & (token_value_df["destination_vault_address"] == "0xa7569A44f348d3D70d8ad5889e50F78E33d80D35")
    & (token_value_df["token_address"] == "0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48")
]

,destination_vault_address,token_address,block,spot_price


In [ ]:
rebalance_event_df

,transactionHash,timestamp,blockNumber,tokenInAmount,tokenOut,destinationInAddress,tokenIn,tokenOutAmount,destinationOutAddress,tokenInAddress,tokenOutAddress
0,0xff8915d0bd9054e7678c4366ae8fd5a0a3cce80943be...,1744988327,22296632,148881.150087,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0xbB2d2dd491204a86ec10a1a6972F940B34fE060e,{'id': '0x4f493b7de8aac7d55f71853688b1f7c8f024...,150000.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x4f493B7dE8aAC7d55F71853688b1F7C8F0243C85,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48
1,0xff3d22a3564c8121f28d7d47feffb8251785ad2c48e4...,1744997471,22297389,148885.474474,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0xbB2d2dd491204a86ec10a1a6972F940B34fE060e,{'id': '0x4f493b7de8aac7d55f71853688b1f7c8f024...,150000.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x4f493B7dE8aAC7d55F71853688b1F7C8F0243C85,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48
2,0xfecf659a8d1c06c45010afc47fcefc6405d906a85e84...,1746235043,22399927,320651.575857,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0xC5c95fCad37E466E25e6ecA1977bbF75C0E1004a,{'id': '0xbeef01735c132ada46aa9aa4c54623caa92a...,347222.682150,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0xBEEF01735c132Ada46AA9aA4c54623cAA92A64CB,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48
3,0xfca3e4ad4e6296b2444ed28a8012cda8694ef5d23ce9...,1746116255,22390138,148075.994800,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0x7876F91BB22148345b3De16af9448081E9853830,{'id': '0x9fb7b4477576fe5b32be4c1843afb1e55f25...,170000.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x9Fb7b4477576Fe5B32be4C1843aFB1e55F251B33,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48
4,0xfc37985d3c7182369be30f9dbe69409a8b05d008bde2...,1745111003,22306807,437134.294459,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0xE4545f9dBC30Ccb6Cda6930DDFd69f3D419FcB61,{'id': '0x5c20b550819128074fd538edf79791733cce...,500000.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x5C20B550819128074FD538Edf79791733ccEdd18,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48
...,...,...,...,...,...,...,...,...,...,...,...
263,0x02a5528289efe28718bed54e9c5b0772f16eba69d609...,1744695875,22272381,76347.504659,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0x7583b1589aDD33320366A48A92794D77763FAE9e,{'id': '0x390f3595bca2df7d23783dfd126427cceb99...,77679.210200,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x390f3595bCa2Df7d23783dFd126427CCeb997BF4,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48
264,0x02065d1b16c7c693afe6358f38647b921a095d6876ec...,1745911055,22373171,26611.724622,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0x7583b1589aDD33320366A48A92794D77763FAE9e,{'id': '0x390f3595bca2df7d23783dfd126427cceb99...,27068.157900,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x390f3595bCa2Df7d23783dFd126427CCeb997BF4,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48
265,0x012e26a2633c10388ffc3ee1a2a95c9569b54ff4f918...,1744556327,22260800,140053.102440,{'id': '0x4f493b7de8aac7d55f71853688b1f7c8f024...,0xC099899d0278CE83976218Cbe58D01dD382dcA32,{'id': '0x57064f49ad7123c92560882a45518374ad98...,149036.479162,0xbB2d2dd491204a86ec10a1a6972F940B34fE060e,0x57064F49Ad7123C92560882a45518374ad982e85,0x4f493B7dE8aAC7d55F71853688b1F7C8F0243C85
266,0x00ff23899cd9bb0075ebe5ac23a3faa815088ddd32ab...,1744906595,22289850,148973.430940,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0xbB2d2dd491204a86ec10a1a6972F940B34fE060e,{'id': '0x4f493b7de8aac7d55f71853688b1f7c8f024...,150000.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x4f493B7dE8aAC7d55F71853688b1F7C8F0243C85,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48


2025-05-05 11:34:46,471 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-05-05 11:34:46,472 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s, %(param_2)s, %(param_3)s, %(param_4)s, %(param_5)s]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class.oid) AND pg_catalog.pg_namespace.nspname != %(nspname_1)s
2025-05-05 11:34:46,472 INFO sqlalchemy.engine.Engine [cached since 344.2s ago] {'table_name': <sqlalchemy.sql.elements.TextClause object at 0x16a447df0>, 'param_1': 'r', 'param_2': 'p', 'param_3': 'f', 'param_4': 'v', 'param_5': 'm', 'nspname_1': 'pg_catalog'}
2025-05-05 11:34:46,473 INFO sqlalchemy.engine.Engine SELECT
    destination_states.*,
    destinations.*
FROM destination_states
JOIN destinations
  ON destination_states.chain_id 

,destination_vault_address,block,chain_id,incentive_apr,fee_apr,base_apr,points_apr,fee_plus_base_apr,total_apr_in,total_apr_out,...,exchange_name,block_deployed,name,symbol,pool_type,pool,underlying,underlying_symbol,underlying_name,denominated_in
0,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20759464,1,0.041235,0.003997,0.005817,0.0,None,0.046281,0.046281,...,balancer,20756405,Tokemak-Wrapped Ether-Balancer rETH Stable Pool,toke-WETH-B-rETH-STABLE,balMetaStable,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,B-rETH-STABLE,Balancer rETH Stable Pool,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2
1,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20766617,1,0.040938,0.003618,0.005802,0.0,None,0.045987,0.045987,...,balancer,20756405,Tokemak-Wrapped Ether-Balancer rETH Stable Pool,toke-WETH-B-rETH-STABLE,balMetaStable,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,B-rETH-STABLE,Balancer rETH Stable Pool,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2
2,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20773761,1,0.040507,0.003295,0.005848,0.0,None,0.045061,0.045061,...,balancer,20756405,Tokemak-Wrapped Ether-Balancer rETH Stable Pool,toke-WETH-B-rETH-STABLE,balMetaStable,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,B-rETH-STABLE,Balancer rETH Stable Pool,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2
3,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20780916,1,0.040196,0.003022,0.005817,0.0,None,0.044440,0.044440,...,balancer,20756405,Tokemak-Wrapped Ether-Balancer rETH Stable Pool,toke-WETH-B-rETH-STABLE,balMetaStable,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,B-rETH-STABLE,Balancer rETH Stable Pool,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2
4,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20788084,1,0.040448,0.002794,0.005885,0.0,None,0.044524,0.044524,...,balancer,20756405,Tokemak-Wrapped Ether-Balancer rETH Stable Pool,toke-WETH-B-rETH-STABLE,balMetaStable,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,0x1E19CF2D73a72Ef1332C882F20534B6519Be0276,B-rETH-STABLE,Balancer rETH Stable Pool,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10843,0xAADf01DD90aE0A6Bb9Eb908294658037096E0404,29590926,8453,0.000000,0.000000,0.000000,0.0,None,0.000000,0.000000,...,tokemak,21241103,baseETH,baseETH,idle,0xAADf01DD90aE0A6Bb9Eb908294658037096E0404,0xAADf01DD90aE0A6Bb9Eb908294658037096E0404,baseETH,baseETH,0x4200000000000000000000000000000000000006
10844,0xAADf01DD90aE0A6Bb9Eb908294658037096E0404,29634126,8453,0.000000,0.000000,0.000000,0.0,None,0.000000,0.000000,...,tokemak,21241103,baseETH,baseETH,idle,0xAADf01DD90aE0A6Bb9Eb908294658037096E0404,0xAADf01DD90aE0A6Bb9Eb908294658037096E0404,baseETH,baseETH,0x4200000000000000000000000000000000000006
10845,0xAADf01DD90aE0A6Bb9Eb908294658037096E0404,29677326,8453,0.000000,0.000000,0.000000,0.0,None,0.000000,0.000000,...,tokemak,21241103,baseETH,baseETH,idle,0xAADf01DD90aE0A6Bb9Eb908294658037096E0404,0xAADf01DD90aE0A6Bb9Eb908294658037096E0404,baseETH,baseETH,0x4200000000000000000000000000000000000006
10846,0xAADf01DD90aE0A6Bb9Eb908294658037096E0404,29720526,8453,0.000000,0.000000,0.000000,0.0,None,0.000000,0.000000,...,tokemak,21241103,baseETH,baseETH,idle,0xAADf01DD90aE0A6Bb9Eb908294658037096E0404,0xAADf01DD90aE0A6Bb9Eb908294658037096E0404,baseETH,baseETH,0x4200000000000000000000000000000000000006


In [ ]:
destination_states_df["underlying_symbol"].value_counts()

underlying_symbol
B-rETH-STABLE           462
wETHrETH                462
rsETH / ETHx            462
rsETH / WETH            462
weETH/ezETH/rswETH      462
ezETH-WETH-BPT          462
weeth-ng                462
osETH/wETH-BPT          462
pxethweth               462
pxsteth                 462
weETH/rETH              462
ethx-f                  462
ETHx/wstETH             462
osETH-rETH              462
ECLP-wstETH-wETH        462
ECLP-wstETH-cbETH       462
wstETH-WETH-BPT         462
frxeth-ng-f             462
pxETH/wETH              462
OETHCRV-f               231
autoETH                 231
balETH                  231
autoLRT                 231
dineroETH               231
rETH-WETH-BPT           183
ECLP-cbETH-wstETH       183
weETH/wETH              183
vAMM-weETH.base/WETH    183
baseETH                 183
Name: count, dtype: int64